# In this notebook, we analyze
- Rates of each gpt label (frames and binary variables) by themselves and w.r.t political leaning
- Jan 6 Case Study: how does coverage on a shared topic differ across political spectrum?
- Segmented frames by "gpt-topic", which was a variable labeled in the dataset already. There is theoretical reason to believe that coverage differs by War/Social Issue/Labor, but we do not find evidence for it


# Future leads
- How GPT-labeled factors predict political leaning 
- How a set of variables (e.g. binaries, outlet political leaning) predict propensity to use one of delegitimizing frames
- dive into residuals
- incorporation of title frame and secondary excerpt frame
- additional dives into topics to compare coverage, e.g. Israel-Hamas


In [11]:
import os

file_path = "full_data.csv"

# Get size in bytes
file_size = os.path.getsize(file_path)
print(f"File size: {file_size} bytes")

File size: 84261234 bytes


In [12]:
import pandas as pd

labels_df = pd.read_csv('labels_df.csv')
df = pd.read_csv('truncated_samples.csv.gz')

labels_df.head(1)


,uuid,title,protest_excerpt,gpt_title_frame,gpt_excerpt_primary_frame,gpt_excerpt_secondary_frame,gpt_demands_present,gpt_protester_quote,gpt_authority_quote,gpt_thematic_present,frames_error,binary_error
0,0057efe4-f7bc-4c05-a7e3-3dcbe2dc8733,Ethiopia declares state of emergency as region...,ETHIOPIA’S AMHARA ETHNIC GROUP ACCUSES BIDEN O...,residual,disruption,harm_to_protesters,no,no,yes,yes,NaN,NaN


In [18]:
counts = pd.concat([
    labels_df['gpt_title_frame'].value_counts().rename('Title'),
    labels_df['gpt_excerpt_primary_frame'].value_counts().rename('Excerpt Primary'),
    labels_df['gpt_excerpt_secondary_frame'].value_counts().rename('Excerpt Secondary')
], axis=1)

counts

,Title,Excerpt Primary,Excerpt Secondary
residual,5244,350,NaN
grievance,930,2441,1846.0
disruption,865,1658,1277.0
protester_violence,628,1907,458.0
harm_to_protesters,551,1145,430.0
impactful,443,782,579.0
widespread_support,397,443,768.0
property_damage,181,403,331.0
peaceful,44,154,313.0


In [14]:
labels_df['gpt_excerpt_secondary_frame'].isna().sum()

np.int64(3281)

# preprocessing: merging GPT labels with original df, transforming binary cols

In [5]:
# df = original dataset
# labels_df = GPT frame annotation results

merged_df = df.merge(
    labels_df.drop(columns=["title", "protest_excerpt"]),
    on="uuid",
    how="left",
    validate="one_to_one"
)


binary_cols = [
    "gpt_demands_present",
    "gpt_protester_quote",
    "gpt_authority_quote",
    "gpt_thematic_present"
]

# change binary to 1s and 0s
merged_df[binary_cols] = (
    merged_df[binary_cols]
    .apply(lambda col: col.str.strip().str.lower())
    .replace({"yes": 1, "no": 0})
)



print("Original rows:", len(df))
print("GPT label rows:", len(labels_df))
print("Merged rows:", len(merged_df))

merged_df.to_csv("full_data.csv", index=False)

Original rows: 9283
GPT label rows: 9283
Merged rows: 9283


In [2]:
import pandas as pd

merged_df = pd.read_csv('full_data.csv')
merged_df.head(1)

,row_id,text-topic,text-topic-exp,text-entity-name,text-entity-gender,text-entity-sentiment,text-entity-sentiment-exp,text-generic-frame,text-generic-frame-exp,text-issue-frame,...,excerpt_length,gpt_title_frame,gpt_excerpt_primary_frame,gpt_excerpt_secondary_frame,gpt_demands_present,gpt_protester_quote,gpt_authority_quote,gpt_thematic_present,frames_error,binary_error
0,1,Ethiopia's Political Instability and State of ...,The article discusses the declaration of a sta...,Ethiopia's Amhara Region,NaN,Negative,The article discusses a state of emergency bei...,"['Security and defense', 'Economic', 'Policy p...",The article discusses a state of emergency dec...,Political Instability and Security Concerns,...,1321,residual,disruption,harm_to_protesters,0,0,1,1,NaN,NaN


## What are the rates of the binary variables, segmented by primary frame? 

In [14]:
binary_cols = [
    "gpt_demands_present",
    "gpt_protester_quote",
    "gpt_authority_quote",
    "gpt_thematic_present"
]


frame_binary_rates = (
    merged_df
    .groupby("gpt_excerpt_primary_frame")[binary_cols]
    .mean()
    .mul(100)
    .round(1)
)

frame_binary_rates

,gpt_demands_present,gpt_protester_quote,gpt_authority_quote,gpt_thematic_present
gpt_excerpt_primary_frame,,,,
disruption,81.7,53.8,78.0,52.8
grievance,98.0,69.7,70.7,66.5
harm_to_protesters,79.4,47.5,78.1,73.2
impactful,74.7,42.2,80.9,74.2
peaceful,74.7,50.0,76.6,44.8
property_damage,76.7,37.5,86.8,65.5
protester_violence,71.5,43.3,87.8,50.8
residual,36.6,20.9,82.0,42.6
widespread_support,85.3,49.2,72.5,63.0


From a descriptive first look:
- residual is anomalously low for demands present (makes sense given grievance was a frame option) and protester quote...
- grievance has by far highest rate of protester quotes
- property damage and protester violence have highest rate of authority quote
- protester_violence is lowest in demands present and is significantly lower than harm to protesters, aside from residual

In [15]:
binary_by_lean = (
    merged_df
    .groupby("political_leaning")[binary_cols]
    .mean()
    .mul(100)
    .round(1)
    .reindex(lean_order)
    .T
)

binary_by_lean

political_leaning,left,left_lean,center,right_lean,right
gpt_demands_present,77.3,82.7,78.4,83.0,80.0
gpt_protester_quote,47.0,56.1,42.4,49.5,51.1
gpt_authority_quote,77.3,77.3,76.2,79.4,82.1
gpt_thematic_present,66.3,59.9,62.7,62.9,59.4


# What about political leaning?

In [21]:
merged_df['political_leaning'].value_counts()

political_leaning
left_lean     4178
right         2441
center        1152
right_lean     921
left           591
Name: count, dtype: int64

In [8]:
lean_order = [
    "left",
    "left_lean",
    "center",
    "right_lean",
    "right"
]

frame_by_lean = pd.crosstab(
    merged_df["gpt_excerpt_primary_frame"],
    merged_df["political_leaning"],
    normalize="columns"
).mul(100).round(1)
# change to normalize="index" for rows sum to 100%

frame_by_lean = frame_by_lean.reindex(columns=lean_order)

frame_by_lean

political_leaning,left,left_lean,center,right_lean,right
gpt_excerpt_primary_frame,,,,,
disruption,8.1,16.3,16.6,17.8,23.6
grievance,22.5,31.4,24.3,27.5,19.0
harm_to_protesters,16.9,12.8,10.4,10.9,11.9
impactful,8.6,8.0,12.2,11.1,6.3
peaceful,2.0,2.0,1.5,0.9,1.3
property_damage,1.7,4.0,3.2,6.3,5.4
protester_violence,28.6,17.4,18.7,17.9,25.9
residual,5.8,2.7,6.6,3.8,3.8
widespread_support,5.8,5.5,6.6,3.9,2.8


Note: columns sum to 100%. 

- We see that disruption rank orders, with more conservative outlets having higher rates.
- protester violence is quite frequent for both extremes of the political spectrum

In [26]:

binary_by_lean = (
    merged_df
    .groupby("political_leaning")[binary_cols]
    .mean()
    .mul(100)
    .round(1)
    .T
)

binary_by_lean = binary_by_lean.reindex(columns=lean_order)

binary_by_lean

political_leaning,left,left_lean,center,right_lean,right
gpt_demands_present,77.3,82.7,78.4,83.0,80.0
gpt_protester_quote,47.0,56.1,42.4,49.5,51.1
gpt_authority_quote,77.3,77.3,76.2,79.4,82.1
gpt_thematic_present,66.3,59.9,62.7,62.9,59.4


less clear trends here, but we do see right outlets have highest authority quote rates. 

# Jan 6 Deep Dive

In [48]:
merged_df.columns

Index(['row_id', 'text-topic', 'text-topic-exp', 'text-entity-name',
       'text-entity-gender', 'text-entity-sentiment',
       'text-entity-sentiment-exp', 'text-generic-frame',
       'text-generic-frame-exp', 'text-issue-frame', 'text-issue-frame-exp',
       'uuid', 'title', 'img-generic-frame', 'img-frame-exp',
       'img-entity-name', 'img-entity-gender', 'img-entity-sentiment',
       'img-entity-sentiment-exp', 'image-url', 'gpt-topic', 'authors',
       'date_publish', 'description', 'language', 'maintext', 'source_domain',
       'url', 'political_leaning', 'month', 'keyword_matches',
       'keyword_match_count', 'word_count', 'label', 'protest_excerpt',
       'excerpt_length', 'gpt_title_frame', 'gpt_excerpt_primary_frame',
       'gpt_excerpt_secondary_frame', 'gpt_demands_present',
       'gpt_protester_quote', 'gpt_authority_quote', 'gpt_thematic_present',
       'frames_error', 'binary_error'],
      dtype='str')

In [8]:
jan6_pattern = r"""(?ix)
\b(
    jan(?:uary)?\.?\s*6(?:th)? |
    january\s+sixth |
    j6 |
    capitol\s+(?:riot|riots|attack|assault|siege|insurrection|breach|storming) |
    (?:riot|riots|attack|assault|siege|insurrection|breach|storming)\s+(?:of|on|at)\s+(?:the\s+)?(?:u\.?s\.?\s+)?capitol |
    storm(?:ed|ing)?\s+(?:the\s+)?(?:u\.?s\.?\s+)?capitol
)\b
"""

text = (
    merged_df["title"].fillna("").astype(str) + " " +
    merged_df["protest_excerpt"].fillna("").astype(str)
)

jan6_df = merged_df[
    text.str.contains(jan6_pattern, regex=True, na=False)
].copy()

print(f"January 6 candidate articles: {len(jan6_df):,}")

# Save all columns
jan6_df.to_csv("jan6_articles.csv", index=False, encoding="utf-8")

print(f"Saved {len(jan6_df):,} articles to jan6_articles.csv")

/var/folders/ll/2pp9305j3zg3b9z2c3vf2ycr0000gp/T/ipykernel_990/3206616480.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text.str.contains(jan6_pattern, regex=True, na=False)


January 6 candidate articles: 1,461
Saved 1,461 articles to jan6_articles.csv


In [ ]:
excerpt_match = df['protest_excerpt'].fillna('').str.contains(
    jan6_pattern, regex=True
)

title_match = df['title'].fillna('').str.contains(
    jan6_pattern, regex=True
)

excerpt_only_df = df[excerpt_match & ~title_match].copy()
both_df = df[excerpt_match & title_match].copy()

print(f"Excerpt-only matches: {len(excerpt_only_df):,}")
print(f"Both Title and Excerpt matches: {len(both_df):,}")

/var/folders/ll/2pp9305j3zg3b9z2c3vf2ycr0000gp/T/ipykernel_79567/2247912804.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  excerpt_match = df['protest_excerpt'].fillna('').str.contains(


Excerpt-only matches: 639
Both Title and Excerpt matches: 803


/var/folders/ll/2pp9305j3zg3b9z2c3vf2ycr0000gp/T/ipykernel_79567/2247912804.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  title_match = df['title'].fillna('').str.contains(


In [39]:
sample = excerpt_only_df[
    ['uuid', 'title', 'protest_excerpt']
].sample(n=30, random_state=42)

sample.to_csv('jan6_sample.csv', index=False)

In [60]:
# Political leaning order
lean_order = [
    "left",
    "left_lean",
    "center",
    "right_lean",
    "right"
]

# Your four binary variables — change names if needed
binary_cols = [
    "gpt_demands_present",
    "gpt_protester_quote",
    "gpt_authority_quote",
    "gpt_thematic_present"
]

Breakdown by primary frame

In [52]:
jan6_excerpt_frames = (
    jan6_df["gpt_excerpt_primary_frame"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .rename("Percent")
    .to_frame()
)

jan6_excerpt_frames["N"] = jan6_df["gpt_excerpt_primary_frame"].value_counts()

jan6_excerpt_frames

,Percent,N
gpt_excerpt_primary_frame,,
protester_violence,62.2,909
residual,11.4,166
disruption,8.1,118
impactful,7.2,105
harm_to_protesters,4.4,65
grievance,2.3,33
property_damage,1.9,28
widespread_support,1.4,21
peaceful,1.1,16


Which outlets report the most about Jan 6?

In [55]:
jan6_lean = (
    jan6_df["political_leaning"]
    .value_counts()
    .reindex(lean_order)
    .rename("N")
    .to_frame()
)

jan6_lean["Percent"] = (
    jan6_lean["N"] / jan6_lean["N"].sum() * 100
).round(1)

jan6_lean

,N,Percent
political_leaning,,
left,203,13.9
left_lean,590,40.4
center,280,19.2
right_lean,128,8.8
right,260,17.8


Unremarkable, given that left_lean is by far the most common in the overall df.

How does excerpt frame usage differ by political leaning?

In [56]:
jan6_frames_by_lean = (
    pd.crosstab(
        jan6_df["gpt_excerpt_primary_frame"],
        jan6_df["political_leaning"],
        normalize="columns"
    )
    .mul(100)
    .round(1)
    .reindex(columns=lean_order)
)

jan6_frames_by_lean

political_leaning,left,left_lean,center,right_lean,right
gpt_excerpt_primary_frame,,,,,
disruption,3.9,6.4,10.4,12.5,10.4
grievance,1.5,1.9,2.9,1.6,3.5
harm_to_protesters,3.4,2.7,2.5,6.2,10.4
impactful,5.9,4.4,8.6,7.0,13.1
peaceful,2.0,0.7,1.1,1.6,1.2
property_damage,0.5,1.9,2.1,0.8,3.5
protester_violence,70.9,72.2,55.4,53.1,44.6
residual,9.9,8.0,15.7,16.4,13.1
widespread_support,2.0,1.9,1.4,0.8,0.4


Interesting-- protester violence frame most used by left and left_lean, while right and right_lean go for harm_to_protesters.

In [70]:
jan6_binary_by_lean = (
    jan6_df
    .groupby("political_leaning")[binary_cols]
    .mean()
    .mul(100)
    .round(1)
    .T
    .reindex(columns=lean_order)
)

jan6_binary_by_lean

political_leaning,left,left_lean,center,right_lean,right
gpt_demands_present,62.6,66.8,63.2,60.2,51.5
gpt_protester_quote,28.1,44.9,32.1,35.2,34.6
gpt_authority_quote,87.2,86.1,85.4,88.3,88.5
gpt_thematic_present,53.2,36.6,42.5,39.8,52.3


Political spectrum seems to not be predictive of quotes (both extremes have similar rates) or thematic (extremes have highest rates), but demands are generally higher for left and left_lean outlets. 

In [27]:
merged_df.columns

Index(['row_id', 'text-topic', 'text-topic-exp', 'text-entity-name',
       'text-entity-gender', 'text-entity-sentiment',
       'text-entity-sentiment-exp', 'text-generic-frame',
       'text-generic-frame-exp', 'text-issue-frame', 'text-issue-frame-exp',
       'uuid', 'title', 'img-generic-frame', 'img-frame-exp',
       'img-entity-name', 'img-entity-gender', 'img-entity-sentiment',
       'img-entity-sentiment-exp', 'image-url', 'gpt-topic', 'authors',
       'date_publish', 'description', 'language', 'maintext', 'source_domain',
       'url', 'political_leaning', 'month', 'keyword_matches',
       'keyword_match_count', 'word_count', 'label', 'protest_excerpt',
       'excerpt_length', 'gpt_title_frame', 'gpt_excerpt_primary_frame',
       'gpt_excerpt_secondary_frame', 'gpt_demands_present',
       'gpt_protester_quote', 'gpt_authority_quote', 'gpt_thematic_present',
       'frames_error', 'binary_error'],
      dtype='str')

In [72]:
merged_df[['gpt_excerpt_primary_frame', 'gpt_excerpt_secondary_frame', 'gpt-topic', 'text-issue-frame', 'text-issue-frame-exp']].head(5)

,gpt_excerpt_primary_frame,gpt_excerpt_secondary_frame,gpt-topic,text-issue-frame,text-issue-frame-exp
0,disruption,harm_to_protesters,Politics,Political Instability and Security Concerns,The article focuses on the declaration of a st...
1,disruption,grievance,Environment,Climate Arsonist,The article frames Rishi Sunak as a 'climate a...
2,grievance,NaN,Education,Religious Freedom and Parental Rights,The article focuses on a group of parents rall...
3,harm_to_protesters,grievance,Crime,Democracy and Free Speech,The article focuses on a judge's ruling to all...
4,grievance,disruption,Politics,Economic Struggle and Government Accountability,The article focuses on widespread anger over t...


# How do War protests differ in coverage from Social Issues/Labor protests?
Boyle et al. finds that War protests receive more critical coverage in headlines by way of being more radical, but not in the article itself. They also do not find evidence that more radical protest types would be more episodic than thematic

In [10]:
with pd.option_context('display.max_rows', None):
    display(merged_df['gpt-topic'].value_counts())

gpt-topic
Politics                     4657
Legal                         682
Labor                         672
Social Issues                 483
Crime                         467
War                           382
Environment                   302
Business                      273
Sports                        203
Education                     187
Entertainment                 183
Immigration                   141
Culture                       119
Health                         60
Technology                     51
Media                          39
Economy                        38
Human Rights                   38
Social                         27
International                  21
no_topic                       19
Society                        16
International Relations        12
Housing                        11
Public Safety                  11
History                        10
Community                       7
Animal Rights                   7
Environmental                   7
Worl

In [89]:
topic_order = ["War", "Social Issues", "Labor", "Legal", "Politics"]

title_frame_by_topic = (
    pd.crosstab(
        merged_df["gpt_title_frame"],
        merged_df["gpt-topic"],
        normalize="columns"
    )
    .mul(100)
    .round(1)
    .reindex(columns=topic_order)
)

title_frame_by_topic

gpt-topic,War,Social Issues,Labor,Legal,Politics
gpt_title_frame,,,,,
disruption,8.4,6.6,13.5,0.6,9.5
grievance,12.8,9.3,18.3,5.1,7.8
harm_to_protesters,3.1,3.9,2.2,10.6,6.1
impactful,1.8,3.3,14.0,5.1,3.6
peaceful,1.3,1.0,0.1,0.1,0.4
property_damage,1.8,2.3,0.0,0.4,2.1
protester_violence,5.0,12.4,0.0,4.5,9.1
residual,60.7,58.4,38.8,72.1,57.4
widespread_support,5.0,2.7,12.9,1.3,4.1


Sample size issue with so many residuals. No clear story

In [90]:
excerpt_primary_by_topic = (
    pd.crosstab(
        merged_df["gpt_excerpt_primary_frame"],
        merged_df["gpt-topic"],
        normalize="columns"
    )
    .mul(100)
    .round(1)
    .reindex(columns=topic_order)
)

excerpt_primary_by_topic

gpt-topic,War,Social Issues,Labor,Legal,Politics
gpt_excerpt_primary_frame,,,,,
disruption,15.2,13.3,21.4,11.4,17.7
grievance,39.8,22.2,43.8,17.6,19.1
harm_to_protesters,7.3,8.9,3.3,18.8,14.3
impactful,3.4,6.6,19.9,10.0,6.5
peaceful,2.1,2.5,0.7,2.1,1.8
property_damage,4.7,5.4,0.0,1.3,4.7
protester_violence,16.5,35.8,0.4,25.5,27.0
residual,1.8,1.9,1.2,10.9,4.3
widespread_support,9.2,3.5,9.2,2.5,4.7


Contrary to what I'd expect, Social Issues more than doubles War in rate of protester violence framing

In [91]:
binary_cols = [
    "gpt_demands_present",
    "gpt_protester_quote",
    "gpt_authority_quote",
    "gpt_thematic_present"
]

binary_by_topic = (
    merged_df
    .groupby("gpt-topic")[binary_cols]
    .mean()
    .mul(100)
    .round(1)
    .T
    .reindex(columns=topic_order)
)

binary_by_topic

gpt-topic,War,Social Issues,Labor,Legal,Politics
gpt_demands_present,88.0,78.9,90.3,70.4,80.4
gpt_protester_quote,53.4,49.1,78.3,34.5,50.7
gpt_authority_quote,80.6,80.1,68.3,81.5,82.2
gpt_thematic_present,62.6,72.5,56.5,52.5,61.1


In [92]:
frames = [
    "protester_violence",
    "harm_to_protesters",
    "property_damage",
    "disruption",
    "grievance",
    "widespread_support",
    "peaceful",
    "impact",
    "residual"
]

combined_results = {}

for topic in topic_order:
    subset = merged_df[merged_df["gpt-topic"] == topic]

    combined_results[topic] = {
        frame: (
            (
                (subset["gpt_excerpt_primary_frame"] == frame) |
                (subset["gpt_excerpt_secondary_frame"] == frame)
            ).mean() * 100
        )
        for frame in frames
    }

excerpt_any_frame_by_topic = (
    pd.DataFrame(combined_results)
    .round(1)
    .reindex(columns=topic_order)
)

excerpt_any_frame_by_topic

,War,Social Issues,Labor,Legal,Politics
protester_violence,23.0,42.7,0.4,33.0,32.3
harm_to_protesters,12.6,12.4,4.6,23.8,19.6
property_damage,8.1,8.5,0.0,3.5,9.1
disruption,25.4,31.5,42.1,22.1,32.3
grievance,63.6,36.6,71.1,24.9,39.6
widespread_support,19.9,8.9,27.2,6.6,11.5
peaceful,6.8,6.6,1.5,5.4,5.4
impact,0.0,0.0,0.0,0.0,0.0
residual,1.8,1.9,1.2,10.9,4.3


This table combines primary and secondary: it does not distinguish between the two. The columns will add up to more than 100%. 

# Does Title Frame reinforce Excerpt Frame?

In [17]:
import pandas as pd
from scipy.stats import chi2_contingency

title_col = "gpt_title_frame"
body_col = "gpt_excerpt_primary_frame"

frame_df = merged_df.dropna(
    subset=[title_col, body_col]
).copy()

# Counts
frame_counts = pd.crosstab(
    frame_df[title_col],
    frame_df[body_col]
)

# Row percentages: excerpt distribution conditional on title frame
frame_pct = pd.crosstab(
    frame_df[title_col],
    frame_df[body_col],
    normalize="index"
).mul(100).round(1)

display(frame_pct)

gpt_excerpt_primary_frame,disruption,grievance,harm_to_protesters,impactful,peaceful,property_damage,protester_violence,residual,widespread_support
gpt_title_frame,,,,,,,,,
disruption,73.9,12.0,1.0,2.3,0.1,3.4,6.6,0.0,0.7
grievance,8.9,78.0,3.1,1.8,1.1,2.0,1.6,0.0,3.4
harm_to_protesters,5.4,0.7,87.8,0.4,0.0,0.4,4.7,0.2,0.4
impactful,7.4,10.4,3.8,70.4,0.2,0.9,5.2,0.0,1.6
peaceful,6.8,34.1,0.0,0.0,50.0,0.0,4.5,0.0,4.5
property_damage,3.3,4.4,0.6,1.7,0.0,75.7,13.8,0.6,0.0
protester_violence,1.4,0.8,2.1,0.8,0.3,5.9,88.7,0.0,0.0
residual,15.5,26.9,11.1,7.4,2.0,3.3,22.7,6.6,4.4
widespread_support,10.1,31.0,2.5,8.3,3.3,1.0,2.8,0.0,41.1


In [20]:
nonres = frame_df[
    frame_df[title_col] != "Residual"
].copy()

# Does title frame equal excerpt primary frame?
nonres["same_frame"] = (
    nonres[title_col] == nonres[body_col]
)

reinforcement = (
    nonres.groupby(title_col)
    .agg(
        N=(body_col, "size"),
        same_frame_rate=("same_frame", "mean")
    )
)

reinforcement["same_frame_rate"] = (
    reinforcement["same_frame_rate"] * 100
).round(1)

reinforcement

,N,same_frame_rate
gpt_title_frame,,
disruption,865,73.9
grievance,930,78.0
harm_to_protesters,551,87.8
impactful,443,70.4
peaceful,44,50.0
property_damage,181,75.7
protester_violence,628,88.7
residual,5244,6.6
widespread_support,397,41.1


Both Harm To Protesters and Protester Violence show extremely high rates of having same title and excerpt frame. I.e. if the excerpt frames protesters as violent or victims of violence, this will communicated in the headline. Sensationalism/clickbait: people believe extremism will get views

In [23]:
from sklearn.metrics import cohen_kappa_score

kappa_all = cohen_kappa_score(
    frame_df[title_col],
    frame_df[body_col]
)

kappa_nonres = cohen_kappa_score(
    nonres[title_col],
    nonres[body_col]
)

print(f"All articles: {kappa_all:.3f}")
print(f"Non-Residual titles: {kappa_nonres:.3f}")

All articles: 0.300
Non-Residual titles: 0.702


In [22]:
from sklearn.metrics import cohen_kappa_score
import pandas as pd

title_col = "gpt_title_frame"
body_col = "gpt_excerpt_primary_frame"

temp = merged_df[[title_col, body_col]].dropna().copy()

# Normalize labels, if needed
for col in [title_col, body_col]:
    temp[col] = temp[col].astype(str).str.strip().str.lower()

nonres = temp[temp[title_col] != "residual"].copy()

for name, df in [("All", temp), ("Non-Residual", nonres)]:
    observed = (df[title_col] == df[body_col]).mean()

    kappa = cohen_kappa_score(
        df[title_col],
        df[body_col]
    )

    expected = (observed - kappa) / (1 - kappa)

    print(
        f"{name}: N={len(df)}, "
        f"Agreement={observed:.3f}, "
        f"Expected={expected:.3f}, "
        f"Kappa={kappa:.3f}"
    )

All: N=9283, Agreement=0.365, Expected=0.092, Kappa=0.300
Non-Residual: N=4039, Agreement=0.752, Expected=0.169, Kappa=0.702


In [24]:
import pandas as pd

title_col = "gpt_title_frame"
primary_col = "gpt_excerpt_primary_frame"
secondary_col = "gpt_excerpt_secondary_frame"

temp = merged_df[
    [title_col, primary_col, secondary_col]
].dropna(subset=[title_col, primary_col]).copy()

# Normalize labels
for col in [title_col, primary_col, secondary_col]:
    temp[col] = temp[col].astype("string").str.strip().str.lower()

# Primary-only agreement
temp["match_primary"] = (
    temp[title_col] == temp[primary_col]
).fillna(False)

# Secondary-only agreement
temp["match_secondary"] = (
    temp[title_col] == temp[secondary_col]
).fillna(False)

# Match either primary or secondary
temp["match_either"] = (
    temp["match_primary"] | temp["match_secondary"]
)

# Secondary rescue: primary doesn't match, but secondary does
temp["secondary_rescue"] = (
    ~temp["match_primary"] & temp["match_secondary"]
)

# Compare all articles vs non-Residual titles
results = []

for name, df in [
    ("All articles", temp),
    ("Non-Residual titles", temp[temp[title_col] != "residual"])
]:
    results.append({
        "Sample": name,
        "N": len(df),
        "Primary match (%)": df["match_primary"].mean() * 100,
        "Primary or Secondary match (%)": df["match_either"].mean() * 100,
        "Secondary rescue (N)": df["secondary_rescue"].sum(),
        "Secondary rescue (%)": df["secondary_rescue"].mean() * 100
    })

agreement_summary = pd.DataFrame(results).round(1)

agreement_summary

,Sample,N,Primary match (%),Primary or Secondary match (%),Secondary rescue (N),Secondary rescue (%)
0,All articles,9283,36.5,43.3,637,6.9
1,Non-Residual titles,4039,75.2,91.0,637,15.8
